# Make figures for the SCA comparison

In [ ]:
import os
from glob import glob
import xarray as xr
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
from tqdm import tqdm

plt.rcParams.update({'font.size': 12, 'font.sans-serif': 'Verdana'})

base_dir = "/Users/rdcrlrka/Research/SIRO/MCS_SCA/"
out_dir = os.path.join(base_dir, "SCA_comparison_results")

# Locate files
files_dict = {
    "PlanetScope": {
        "mosaic_files": sorted(glob(os.path.join(base_dir, "PSS_image_mosaics_clipped", "*.tif"))),
        "SCA_files": sorted(glob(os.path.join(base_dir, "PSS_SCA_maps", "*.tif"))),
        "SCA_regrid_files": sorted(glob(os.path.join(out_dir, "PlanetScope*.nc"))),
        "color": "#332288",
        "shape": "*"
    },
    "HMS-EB": {
        "SCA_files": sorted(glob(os.path.join(base_dir, "model_SCA_maps", "HMS-EB*.nc"))),
        "color": "#882255",
        "shape": "o"
    },
    "HMS-TI": {
        "SCA_files": sorted(glob(os.path.join(base_dir, "model_SCA_maps", "HMS-TI*.nc"))),
        "color": "#CC6677",
        "shape": "s"
    },
    "iSnobal": {
        "SCA_files": sorted(glob(os.path.join(base_dir, "model_SCA_maps", "iSnobal*.nc"))),
        "color": "#88CCEE",
        "shape": "^"
    },
    "SnowModel": {
        "SCA_files": sorted(glob(os.path.join(base_dir, "model_SCA_maps", "SnowModel*.nc"))),
        "color": "#117733",
        "shape": "h"
    }
}

print("Located files:")
for key in files_dict.keys():
    print(key)
    for sub_key in files_dict[key].keys():
        print(f"\t{sub_key}:", len(files_dict[key][sub_key]))



## Define some color palettes

In [ ]:
sca_cmap = matplotlib.colors.ListedColormap(["#e7e6e6", "#2A96BD"])
sca_cmap

In [ ]:
ds_colors = [files_dict[key]["color"] for key in files_dict.keys()]
ds_cmap = matplotlib.colors.ListedColormap(ds_colors)
ds_cmap


## Map timeseries GIF

In [ ]:
# Load SCA totals timeseries
sca_total_file = os.path.join(out_dir, "SCA_totals_timeseries_compiled.csv")
sca_total_df = pd.read_csv(sca_total_file, index_col=0)
sca_total_df.index = pd.DatetimeIndex(sca_total_df.index)

# Preprare the range of PlanetScope SCA (accounting for masked area and resolution)
pss_sca_cols = [col for col in sca_total_df.columns if col.startswith("SCA_PlanetScope") or col.startswith("SCA_PSS-")]
pss_mask_cols = [col for col in sca_total_df.columns if col.startswith("masked_area_PlanetScope") or col.startswith("masked_area_PSS-")]
pss_sca_cols = sorted(pss_sca_cols)
pss_mask_cols = sorted(pss_mask_cols)
pss_sca_arr = np.stack([sca_total_df[col].values for col in pss_sca_cols], axis=1)
pss_mask_arr = []
for col in pss_mask_cols:
    arr = sca_total_df[col].values
    arr_adj = arr - np.nanmin(arr)
    pss_mask_arr.append(arr_adj)
pss_mask_arr = np.stack(pss_mask_arr, axis=1)

pss_sca_plus_mask = pss_sca_arr + pss_mask_arr

pss_sca_min = np.nanmin(pss_sca_arr, axis=1)
pss_sca_max = np.nanmax(pss_sca_plus_mask, axis=1)
pss_sca_mean = np.nanmean(pss_sca_arr, axis=1)

# Add to SCA df
sca_total_df["PlanetScope_SCA_min_m2"] = pss_sca_min
sca_total_df["PlanetScope_SCA_max_m2"] = pss_sca_max
sca_total_df["PlanetScope_SCA_mean_m2"] = pss_sca_mean

# Plot a preview
plt.figure(figsize=(15, 7))
# models
for ds in ["HMS-EB", "HMS-TI", "iSnobal", "SnowModel"]:
    sca_col = f"SCA_{ds}_m2"
    mask_col = f"masked_area_{ds}_m2"
    if sca_col in sca_total_df.columns and mask_col in sca_total_df.columns:
        sca = sca_total_df[sca_col] / 1e6  # km^2
        plt.plot(
            sca_total_df.index, sca, f"{files_dict[ds]["shape"]}-",
            color=files_dict[ds]["color"], label=ds
            )

# PlanetScope envelope
plt.errorbar(
    sca_total_df.index, 
    (pss_sca_max + pss_sca_min)/2 /1e6,
    yerr=(pss_sca_max-pss_sca_min)/ 1e6, linestyle='-',
    color=files_dict["PlanetScope"]["color"], label="PlanetScope range"
    )

plt.xlabel("Date")
plt.ylabel("Snow-Covered Area (km²)")
plt.title("Snow-Covered Area Time Series")
plt.legend(loc='right', bbox_to_anchor=[1.1, 0.4, 0.2, 0.2])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
gif_out_dir = os.path.join(out_dir, "SCA_gif")
os.makedirs(gif_out_dir, exist_ok=True)

def process_date(
        date, pss_mosaic_file, pss_sca_file, 
        hms_eb_sca_file, hms_ti_sca_file, is_sca_file, sm_sca_file,
        sca_total_df, files_dict, gif_out_dir, sca_cmap,
        ):

    dt = np.datetime64(date)
    fig_file = os.path.join(gif_out_dir, f"{date}_SCA_comparison.png")
    if os.path.exists(fig_file):
        return

    # Load model SCA maps for this process
    with (
        xr.open_dataset(hms_eb_sca_file).SCA.squeeze() as hms_eb_sca,
        xr.open_dataset(hms_ti_sca_file).SCA.squeeze() as hms_ti_sca,
        xr.open_dataset(is_sca_file).SCA.squeeze() as is_sca,
        xr.open_dataset(sm_sca_file).SCA.squeeze() as sm_sca,
        rxr.open_rasterio(pss_mosaic_file, masked=True).squeeze() as pss_mosaic,
        rxr.open_rasterio(pss_sca_file, masked=True).squeeze() as pss_sca
    ):

        pss_mosaic = pss_mosaic / 1e4

        # select model SCAs at nearest time
        hms_eb_sca_samp = hms_eb_sca.sel(time=dt, method='nearest')
        hms_ti_sca_samp = hms_ti_sca.sel(time=dt, method='nearest')
        is_sca_samp = is_sca.sel(time=dt, method='nearest')
        sm_sca_samp = sm_sca.sel(time=dt, method='nearest')

        # plot
        gs = matplotlib.gridspec.GridSpec(3, 3, height_ratios=[2,2,1])
        fig = plt.figure(figsize=(12,12))
        ax = [
            fig.add_subplot(gs[0,0]), fig.add_subplot(gs[0,1]), fig.add_subplot(gs[0,2]),
            fig.add_subplot(gs[1,0]), fig.add_subplot(gs[1,1]), fig.add_subplot(gs[1,2]),
            fig.add_subplot(gs[2,:]),
        ]

        # RGB
        ax[0].imshow(
            np.dstack([pss_mosaic.isel(band=2), pss_mosaic.isel(band=1), pss_mosaic.isel(band=0)]), clim=(0,1),
            extent=(min(pss_mosaic.x)/1e3, max(pss_mosaic.x)/1e3, min(pss_mosaic.y)/1e3, max(pss_mosaic.y)/1e3)
        )
        ax[0].set_title("PlanetScope RGB")
        
        # SCA maps
        for sca_map, axis, title in zip(
            [pss_sca, is_sca_samp, sm_sca_samp, hms_eb_sca_samp, hms_ti_sca_samp], 
            ax[1:], 
            ["PlanetScope SCA", "iSnobal SCA", "SnowModel SCA", "HMS-EB SCA", "HMS-TI SCA"]
        ):
            # mask nodata values
            sca_map = xr.where(sca_map!=255, sca_map, np.nan)
            # SnowModel is flipped
            if title=="SnowModel SCA":
                sca_data = np.flipud(sca_map.data)
            else:
                sca_data = sca_map.data

            im = axis.imshow(
                sca_data, cmap=sca_cmap, clim=(0,1),
                extent=(min(sca_map.x)/1e3, max(sca_map.x)/1e3, min(sca_map.y)/1e3, max(sca_map.y)/1e3)
            )
            axis.set_title(title)

        # SCA time series
        date_mask = sca_total_df.index <= dt
        sca_total_plot_df = sca_total_df[date_mask]
        # models
        for model in list(files_dict.keys())[1:]:
            sca = sca_total_plot_df[f"SCA_{model}_m2"]
            ax[6].plot(
                sca.index, sca.values / 1e6, files_dict[model]["shape"], linestyle='-',
                color=files_dict[model]["color"], label=model
            )
        # PlanetScope
        pss_mean = (sca_total_plot_df["PlanetScope_SCA_max_m2"] + sca_total_plot_df["PlanetScope_SCA_min_m2"]) / 2
        pss_err = (sca_total_plot_df["PlanetScope_SCA_max_m2"] - sca_total_plot_df["PlanetScope_SCA_min_m2"])
        ax[6].errorbar(
            sca_total_plot_df.index, pss_mean / 1e6, yerr= pss_err / 1e6, 
            linestyle='-', color=files_dict["PlanetScope"]["color"], label="PlanetScope range", linewidth=2
        )
        ax[6].set_xlim(np.datetime64("2022-09-15"), np.datetime64("2025-07-15"))
        ax[6].set_ylim(-50, 1350)
        ax[6].set_xlabel("Date")
        ax[6].set_ylabel("Snow-covered area [km$^2$]")
        ax[6].legend(loc='center right', bbox_to_anchor=[1.05, 0.4, 0.2, 0.2])
        ax[6].grid(alpha=0.3)

        # axes labels
        for axis in [ax[0], ax[3]]:
            axis.set_ylabel("Northing [km]")
        for axis in ax[3:6]:
            axis.set_xlabel("Easting [km]")

        # colorbar
        cax = fig.add_axes([0.93, 0.5, 0.02, 0.2])
        cbar = fig.colorbar(im, cax=cax, orientation='vertical')
        cbar.set_ticks([0.25, 0.75])
        cbar.set_ticklabels(["No snow", "Snow"])

        fig.suptitle(date)

        # Save to file
        fig.savefig(fig_file, dpi=250, bbox_inches='tight')
        plt.close()

# Prepare argument list for parallel processing
dates = []
pss_mosaic_files = []
pss_sca_files = []
for pss_mosaic_file in files_dict["PlanetScope"]["mosaic_files"]:
    date = os.path.basename(pss_mosaic_file).split('_')[0]
    # get corresponding SCA file
    pss_sca_file = [x for x in files_dict["PlanetScope"]["SCA_files"] if date in os.path.basename(x)][0]
    dates.append(date)
    pss_mosaic_files.append(pss_mosaic_file)
    pss_sca_files.append(pss_sca_file)

# Model SCA files
# NOTE: Found no difference in SCA by varying SWE threshold, so just using the first SCA map for each model
hms_eb_sca_file = files_dict["HMS-EB"]["SCA_files"][0]
hms_ti_sca_file = files_dict["HMS-TI"]["SCA_files"][0]
is_sca_file = files_dict["iSnobal"]["SCA_files"][0]
sm_sca_file = files_dict["SnowModel"]["SCA_files"][0]

# Prepare argument tuples
for date, pss_mosaic_file, pss_sca_file in tqdm(list(zip(dates, pss_mosaic_files, pss_sca_files))):
    process_date(
        date, pss_mosaic_file, pss_sca_file, 
        hms_eb_sca_file, hms_ti_sca_file, is_sca_file, sm_sca_file,
        sca_total_df, files_dict, gif_out_dir, sca_cmap
        )



## Confusion matrices

## Variation with terrain

In [ ]:
terrain_bin_files = sorted(glob(os.path.join(out_dir, "recall*.nc")))

def plot_recall_polar(recall_ds, cmap='viridis', ax=None):
    recall = recall_ds['recall'].mean(dim='time')
    elev_bins = recall_ds['recall'].attrs['elev_bins']
    aspect_bins = recall_ds['recall'].attrs['aspect_bins']

    n_elev_bins = len(elev_bins) - 1
    n_aspect_bins = len(aspect_bins) - 1

    # Bin centers
    elev_centers = 0.5 * (elev_bins[:-1] + elev_bins[1:])
    aspect_centers = 0.5 * (aspect_bins[:-1] + aspect_bins[1:])

    # Meshgrid for plotting
    r, theta = np.meshgrid(elev_centers, np.deg2rad(aspect_centers), indexing='ij')

    # Plot
    # pcolormesh expects (n_r+1, n_theta+1) for bin edges, so we use centers for imshow
    c = ax.pcolormesh(theta, r, recall, cmap=cmap, shading='auto')
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    plt.colorbar(c, ax=ax, label='Recall', shrink=0.5)
    ax.set_xlabel('Aspect')
    ax.set_ylabel('Elevation')
    
fig, ax = plt.subplots(2, 2, subplot_kw={'projection': 'polar'}, figsize=(10, 10))
ax = ax.flatten()
for i, file in enumerate(terrain_bin_files):
    recall_ds = xr.open_dataset(file)
    plot_recall_polar(recall_ds, ax=ax[i])
    model_name = os.path.splitext(os.path.basename(file))[0].split('_')[-1]
    ax[i].set_title(model_name)

plt.show()


In [ ]:
terrain_bin_files[0]